In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats

BASE_DIR = Path(r"D:\Shikha\bias_project")
EMB      = BASE_DIR / "outputs" / "embeddings"
GSS_DIR  = BASE_DIR / "validation_data" / "gss" / "processed"

PERIODS = ["modern_2020", "modern_2024"]   # both modern cells GSS can validate

# Period-split _v2 scores: (file, score column)
methods = {
    "w2v":     (EMB / "word2vec" / "weat6_scores_word2vec_v2.csv", "weat6_pairs_d"),
    "glove":   (EMB / "glove"    / "weat6_scores_glove_v2.csv",    "weat6_d"),
    "bert":    (EMB / "bert"     / "seat_scores_bert_v2.csv",      "seat_career_family"),
    "roberta": (EMB / "roberta"  / "seat_scores_roberta.csv",      "seat_career_family"),
}

# Build long table: state, period, method, score
frames = []
for name, (path, col) in methods.items():
    df = pd.read_csv(path)
    df = df[df["period"].isin(PERIODS)][["state", "period", col]].rename(columns={col: "score"})
    df["method"] = name
    frames.append(df)
    for p in PERIODS:
        print(f"{name:<8} {p:<13}: {(df['period']==p).sum()} states")

long = pd.concat(frames, ignore_index=True)
print(f"\nLong table: {len(long)} rows ({long['method'].nunique()} methods x {len(PERIODS)} periods)")
print(long.head(8).to_string(index=False))

w2v      modern_2020  : 27 states
w2v      modern_2024  : 44 states
glove    modern_2020  : 32 states
glove    modern_2024  : 44 states
bert     modern_2020  : 50 states
bert     modern_2024  : 49 states
roberta  modern_2020  : 50 states
roberta  modern_2024  : 49 states

Long table: 345 rows (4 methods x 2 periods)
state      period   score method
   AL modern_2020 -0.3096    w2v
   AR modern_2020 -0.5345    w2v
   CA modern_2020 -0.5263    w2v
   CT modern_2020 -0.3267    w2v
   DE modern_2020  1.2586    w2v
   FL modern_2020  0.2583    w2v
   GA modern_2020 -0.0668    w2v
   HI modern_2020 -0.1353    w2v


In [2]:
STATE_TO_REGION = {
    'CT':1,'ME':1,'MA':1,'NH':1,'RI':1,'VT':1,
    'NJ':2,'NY':2,'PA':2,
    'IL':3,'IN':3,'MI':3,'OH':3,'WI':3,
    'IA':4,'KS':4,'MN':4,'MO':4,'NE':4,'ND':4,'SD':4,
    'DE':5,'FL':5,'GA':5,'MD':5,'NC':5,'SC':5,'VA':5,'WV':5,
    'AL':6,'KY':6,'MS':6,'TN':6,
    'AR':7,'LA':7,'OK':7,'TX':7,
    'AZ':8,'CO':8,'ID':8,'MT':8,'NV':8,'NM':8,'UT':8,'WY':8,
    'AK':9,'CA':9,'HI':9,'OR':9,'WA':9
}

gss = pd.read_csv(GSS_DIR / "gss_region_year_clean.csv")
gss_recent = (gss[gss['year'] >= 2010]
              .groupby('region')['composite_score'].mean()
              .reset_index().rename(columns={'composite_score': 'gss_attitude'}))

long["region"] = long["state"].map(STATE_TO_REGION)
val = long.merge(gss_recent, on="region")

print("=== GSS Validation by period (clean period-split scores) ===")
print("GSS composite: HIGHER = LESS traditional. Expected: NEGATIVE r.\n")
print(f"{'Period':<13} {'Method':<9} {'n':>4} {'Pearson r':>11} {'p':>8} {'Spearman':>10} {'p':>8}")
print("-" * 66)

rows = []
for period in PERIODS:
    for name in ["w2v", "glove", "bert", "roberta"]:
        sub = val[(val["period"] == period) & (val["method"] == name)][["score", "gss_attitude"]].dropna()
        if len(sub) < 4:
            print(f"{period:<13} {name:<9} {len(sub):>4}  (too few states)")
            continue
        pr, pp = stats.pearsonr(sub["score"], sub["gss_attitude"])
        sr, sp = stats.spearmanr(sub["score"], sub["gss_attitude"])
        print(f"{period:<13} {name:<9} {len(sub):>4} {pr:>11.3f} {pp:>8.3f} {sr:>10.3f} {sp:>8.3f}")
        rows.append({"period": period, "method": name, "n": len(sub),
                     "pearson_r": round(pr,4), "pearson_p": round(pp,4),
                     "spearman_r": round(sr,4), "spearman_p": round(sp,4)})

VAL_DIR = BASE_DIR / "outputs" / "validation"
VAL_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(rows).to_csv(VAL_DIR / "gss_validation_by_period.csv", index=False)
val.to_csv(VAL_DIR / "gss_validation_merged.csv", index=False)
print(f"\nSaved: {VAL_DIR / 'gss_validation_by_period.csv'}")
print("Note: region-level GSS (9 values across ~50 states) limits power at demo scale.")

=== GSS Validation by period (clean period-split scores) ===
GSS composite: HIGHER = LESS traditional. Expected: NEGATIVE r.

Period        Method       n   Pearson r        p   Spearman        p
------------------------------------------------------------------
modern_2020   w2v         27      -0.196    0.328     -0.211    0.291
modern_2020   glove       32       0.079    0.669      0.008    0.966
modern_2020   bert        50      -0.052    0.722     -0.092    0.523
modern_2020   roberta     50       0.039    0.786     -0.027    0.854
modern_2024   w2v         44      -0.080    0.606     -0.095    0.538
modern_2024   glove       44      -0.088    0.571     -0.051    0.740
modern_2024   bert        49      -0.057    0.696     -0.086    0.556
modern_2024   roberta     49       0.027    0.854      0.016    0.912

Saved: D:\Shikha\bias_project\outputs\validation\gss_validation_by_period.csv
Note: region-level GSS (9 values across ~50 states) limits power at demo scale.
